# Day 2 — Single-cell spatial with Xenium

**Dataset:** a public 10x Xenium *non-diseased human kidney* section (~97,560 cells, 377-gene Multi-Tissue panel).

**You will learn to**
- read a Xenium section without exhausting memory;
- separate control features and run imaging-specific QC;
- cluster and annotate cell types on a small panel;
- choose a spatial graph and run single-cell spatial statistics.

> **Memory.** We load only the cell-by-feature matrix and the cell table. We do not load `transcripts.parquet` or the morphology images; that is what exceeds 25 GB. One section at a time.

### Setup (run first)

On the course servers this does nothing, since the packages are pre-installed. On Google Colab or a fresh laptop it installs what is missing. Data downloads automatically the first time it is needed, except the Xenium bundle on Day 2 (see the README).

In [ ]:
import importlib, subprocess, sys
_req = [('scanpy','scanpy'), ('squidpy','squidpy'), ('leidenalg','leidenalg'),
        ('igraph','igraph'), ('pyclustree','pyclustree'),
        ('scikit-image','skimage'), ('imageio','imageio')]
_missing = [pip for pip, mod in _req if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=False)
print('environment ready')

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.titlesize": 11,
    "font.weight": "normal",
    "figure.dpi": 110,
})

def _data_dir():
    env = os.environ.get("SPATIAL_COURSE_DATA")
    if env:
        return Path(env)
    shared = Path("/home/shared/spatial_course_data")
    if shared.exists():
        return shared
    local = Path.home() / "spatial_course_data"
    local.mkdir(parents=True, exist_ok=True)
    return local

DATA_DIR = _data_dir()
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

def load_visium():
    p = DATA_DIR / "visium_lymph_node.h5ad"
    if p.exists():
        adata = sc.read_h5ad(p)
    else:
        adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
        try:
            adata.write(p)
        except Exception:
            pass
    adata.var_names_make_unique()
    return adata

def load_seqfish():
    p = DATA_DIR / "seqfish_embryo.h5ad"
    if p.exists():
        return sc.read_h5ad(p)
    adata = sq.datasets.seqfish()
    try:
        adata.write(p)
    except Exception:
        pass
    return adata

def xenium_dir():
    d = DATA_DIR / "xenium_kidney"
    if (d / "cell_feature_matrix.h5").exists():
        return d
    raise FileNotFoundError(
        f"Xenium bundle not found in {d}. See the README (Data): download the 10x "
        "Xenium Output Bundle once and unzip it there, or run scripts/prefetch_data.py."
    )

print("data dir:", DATA_DIR)

## 1. Reading Xenium

> **Method note — lean vs full.** `spatialdata_io.xenium(path)` builds a full `SpatialData` object with images, shapes, and the transcript table, which is ideal for visualisation but heavy. For table-level analysis we read `cell_feature_matrix.h5` and `cells.parquet` straight into AnnData. Same cells, a fraction of the memory.

In [ ]:
xpath = xenium_dir()
adata = sc.read_10x_h5(xpath / 'cell_feature_matrix.h5')
adata.var_names_make_unique()
cells_pq = xpath / 'cells.parquet'
cells = pd.read_parquet(cells_pq) if cells_pq.exists() else pd.read_csv(xpath / 'cells.csv.gz')
cells = cells.set_index('cell_id')
cells.index = cells.index.astype(str)
adata.obs_names = adata.obs_names.astype(str)
adata.obs = adata.obs.join(cells)
adata.obsm['spatial'] = adata.obs[['x_centroid', 'y_centroid']].to_numpy()
adata

## 2. Control features

> **Method note — built-in controls.** Negative control probes target no real transcript and estimate non-specific binding. Negative control codewords (blanks) are valid codes assigned to no gene and estimate the decoding error rate. A healthy run puts only a small fraction of counts on these. We record a per-cell control fraction, then keep only the real genes.

In [ ]:
is_gene = adata.var['feature_types'] == 'Gene Expression'
print(adata.var['feature_types'].value_counts())
adata.obs['total_counts_all'] = np.asarray(adata.X.sum(1)).ravel()
adata.obs['control_counts'] = np.asarray(adata[:, (~is_gene).values].X.sum(1)).ravel()
adata = adata[:, is_gene.values].copy()

## 3. Quality control

> **Method note — Xenium metrics.** Transcripts and genes per cell flag empty or fragmented segments; control fraction flags noise; cell area flags bad segmentation (very small or very large). On a 377-gene panel the absence of a gene is weak evidence.

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)
adata.obs['control_frac'] = adata.obs['control_counts'] / adata.obs['total_counts_all'].clip(lower=1)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 3))
axs[0].hist(adata.obs['total_counts'], bins=80); axs[0].set_title('transcripts / cell')
axs[1].hist(adata.obs['n_genes_by_counts'], bins=80); axs[1].set_title('genes / cell')
axs[2].hist(adata.obs['control_frac'], bins=80); axs[2].set_title('control fraction')
axs[3].hist(adata.obs['cell_area'], bins=80); axs[3].set_title('cell area')
plt.tight_layout(); plt.show()

In [ ]:
sq.pl.spatial_scatter(adata, color='total_counts', shape=None, size=2)

> **Exercise 1.** Choose a minimum transcripts-per-cell threshold. On a copy, filter, then plot the *removed* cells in space. Do they cluster in a region (suspect tissue or segmentation) or scatter (random low-quality cells)?

**Solution.**

In [ ]:
adata_q = adata.copy()
sc.pp.filter_cells(adata_q, min_counts=30)
removed = adata[~adata.obs_names.isin(adata_q.obs_names)].copy()
print(removed.n_obs, 'cells removed')
sq.pl.spatial_scatter(removed, color='total_counts', shape=None, size=4)

## 4. Filter and normalise (worked spine)

> **Method note — small panels.** With only 377 genes we skip highly variable gene selection: there are too few genes to drop any without losing signal. Total-count normalisation then `log1p` is the common default. An alternative divides each cell's counts by its segment area, $\tilde{x}_{ig} = x_{ig}/a_i$, assuming counts scale with captured cell volume; this helps when segmentation yields very different cell sizes. We use total-count here and keep raw counts in a layer.

In [ ]:
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata

## 5. Embed and cluster

On ~10^5 cells this is a few minutes on CPU.

In [ ]:
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_neighbors=15)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1.0, flavor='igraph', n_iterations=2, directed=False)

In [ ]:
sc.pl.umap(adata, color=['leiden', 'total_counts'])

## 6. Annotate cell types

> **Method note — typing on a small panel, and how scoring works.** Fewer genes make clusters noisier and rare types harder to separate. `score_genes` computes, per cell, the mean expression of a marker set minus the mean of a random reference set matched on expression level, $s = \bar{x}_{\text{markers}} - \bar{x}_{\text{control}}$, so the score is high only when the markers are specifically elevated. We average scores per cluster and take the argmax, which assumes each cluster is dominated by one type; doublet or split clusters break that, so refine by eye. Reference label transfer (CellTypist, scANVI, `ingest`) scales when a matched reference exists, but inherits any segmentation contamination.

In [ ]:
markers = {
    'PT': ['LRP2', 'CUBN', 'SLC34A1'],
    'TAL': ['UMOD', 'SLC12A1'],
    'DCT': ['SLC12A3'],
    'PC': ['AQP2'],
    'IC': ['SLC4A1', 'SLC26A4'],
    'Podocyte': ['NPHS1', 'NPHS2', 'PODXL'],
    'Endothelial': ['PECAM1', 'FLT1', 'EMCN'],
    'Fibroblast': ['PDGFRB', 'COL1A1', 'DCN'],
    'T': ['CD3D', 'CD3E', 'CD2'],
    'B': ['MS4A1', 'CD79A'],
    'Myeloid': ['LYZ', 'CD68', 'ITGAM'],
}
markers = {k: [g for g in v if g in adata.var_names] for k, v in markers.items()}
markers = {k: v for k, v in markers.items() if v}

In [ ]:
sc.pl.dotplot(adata, markers, groupby='leiden')

In [ ]:
for ct, genes in markers.items():
    sc.tl.score_genes(adata, genes, score_name=f'score_{ct}')
score_cols = [f'score_{ct}' for ct in markers]
cluster_scores = adata.obs.groupby('leiden', observed=True)[score_cols].mean()
mapping = cluster_scores.idxmax(axis=1).str.replace('score_', '', regex=False).to_dict()
adata.obs['celltype'] = adata.obs['leiden'].map(mapping).astype('category')
mapping

> **Exercise 2.** Read the dotplot and correct the automatic mapping where it is wrong (for example a cluster split between two types, or an immune cluster missed). Edit `mapping` for one or two clusters, reassign `celltype`, and replot in space.

**Solution.**

In [ ]:
mapping['3'] = 'Myeloid'
adata.obs['celltype'] = adata.obs['leiden'].map(mapping).astype('category')
sq.pl.spatial_scatter(adata, color='celltype', shape=None, size=2)

In [ ]:
sq.pl.spatial_scatter(adata, color='celltype', shape=None, size=2)

## 7. Spatial visualization

> **Method note — read the right map.** At single-cell resolution we plot points rather than spots, with `sq.pl.spatial_scatter(..., shape=None)`. The same habits apply: a **continuous** overlay (a gene, a QC metric) shows gradients and intensity, a **categorical** overlay (cell types) shows tissue architecture; tune `size` and `alpha` so dense regions do not saturate; **crop** to a coordinate box to read fine structure (a glomerulus, a tubule cross-section); and `groups=` spotlights one or two types. Look for contiguous compartments (cortex vs medulla, a tubule wall), smooth gradients, or scattered cells (a state, or segmentation noise).

In [ ]:
viz_genes = [g for g in ['UMOD', 'LRP2', 'PECAM1', 'NPHS2', 'CD3D'] if g in adata.var_names]
sq.pl.spatial_scatter(adata, color=viz_genes[0], shape=None, size=2)

In [ ]:
sq.pl.spatial_scatter(adata, color=viz_genes[:4], shape=None, size=2)

In [ ]:
sq.pl.spatial_scatter(adata, color=['celltype', 'total_counts'], shape=None, size=2)

In [ ]:
sq.pl.spatial_scatter(adata, color='total_counts', shape=None, size=3, alpha=0.5)

In [ ]:
x = adata.obs['x_centroid']
y = adata.obs['y_centroid']
xa, xb = x.quantile(0.45), x.quantile(0.6)
ya, yb = y.quantile(0.45), y.quantile(0.6)
sub = adata[x.between(xa, xb) & y.between(ya, yb)].copy()
print(sub.n_obs, 'cells in crop')
sq.pl.spatial_scatter(sub, color='celltype', shape=None, size=8)

In [ ]:
top_types = adata.obs['celltype'].value_counts().index[:3].tolist()
sq.pl.spatial_scatter(adata, color='celltype', groups=top_types, shape=None, size=2)

> **Exercise (visualization).** Plot four genes of your choice as a grid; crop to a coordinate box and describe the local architecture; and overlay a single cell type on the section.

**Solution.**

In [ ]:
genes = [g for g in ['UMOD', 'LRP2', 'PECAM1', 'CD3D'] if g in adata.var_names]
sq.pl.spatial_scatter(adata, color=genes, shape=None, size=2)
one = adata.obs['celltype'].value_counts().index[0]
adata.obs['one_type'] = (adata.obs['celltype'] == one).astype('category')
sq.pl.spatial_scatter(adata, color='one_type', shape=None, size=2)

**Answer (example).** A coordinate-box crop usually exposes anatomy that the whole-section view blurs: tubule cross-sections, a glomerulus, or a vascular track. A type that fills a connected region is a compartment; one sprinkled everywhere is a state or a segmentation artefact.

## 8. Spatial statistics

> **Method note — which graph, and what enrichment reports.** kNN gives every cell the same number of neighbours; Delaunay follows tissue geometry and avoids long-range links in sparse areas; a radius graph fixes a physical distance. The choice changes results, so state it. Neighbourhood enrichment then reports a permutation z-score $z = (O-\mu_\pi)/\sigma_\pi$ on the observed cell-type adjacency $O$, so it measures spatial co-location at the graph's scale, not abundance.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type='generic', n_neighs=6)
sq.gr.nhood_enrichment(adata, cluster_key='celltype')
sq.pl.nhood_enrichment(adata, cluster_key='celltype', figsize=(6, 6))

> **Exercise 3.** Rebuild the graph with Delaunay on a copy, rerun neighbourhood enrichment, and compare. Which contacts strengthen or weaken when the graph changes?

**Solution.**

In [ ]:
ad = adata.copy()
sq.gr.spatial_neighbors(ad, coord_type='generic', delaunay=True)
sq.gr.nhood_enrichment(ad, cluster_key='celltype')
sq.pl.nhood_enrichment(ad, cluster_key='celltype', figsize=(6, 6))

Co-occurrence asks how the chance of finding one type near another changes with distance. For types $A,B$ at distance $d$ it is the ratio $\dfrac{p(B \mid A, d)}{p(B)}$: values above 1 mean $B$ is enriched near $A$ at that range, separating tight contact from loose regional association.

In [ ]:
sq.gr.co_occurrence(adata, cluster_key='celltype')
focus = adata.obs['celltype'].value_counts().index[0]
sq.pl.co_occurrence(adata, cluster_key='celltype', clusters=focus, figsize=(7, 4))
print('focus:', focus)

> **Reading co-occurrence.** The curve is the ratio $p(B \mid A, d)/p(B)$ against distance $d$. Above 1 means type $B$ is enriched around $A$ at that range; at 1 there is no association; below 1 is depletion. A peak at short distance is direct contact; a plateau above 1 out to long range is broad regional co-occurrence; a curve that starts high and decays marks a gradient away from $A$.

In [ ]:
sq.gr.spatial_autocorr(adata, mode='moran')
adata.uns['moranI'].head(10)

> **Exercise 4.** Plot the top spatially variable genes in space. What structure does each mark (a tubule type, vasculature, an immune patch)? Name one and say what it tells you about tissue organisation.

**Solution.**

In [ ]:
top = adata.uns['moranI'].head(4).index.tolist()
sq.pl.spatial_scatter(adata, color=top, shape=None, size=2)

**Answer.** The top spatially variable genes by Moran's I are UMOD (thick ascending limb), AQP2 (collecting-duct principal cells), MYH11 (vascular smooth muscle around vessels), and KNG1 and GATM (proximal tubule). Each traces a kidney compartment: UMOD and AQP2 mark nephron segments, MYH11 marks the vasculature, and KNG1/GATM mark the cortical proximal tubule. They map anatomy (cortex/medulla, tubules, vessels), not a dispersed state — note that the canonical PT markers in the marker dict (LRP2/CUBN/SLC34A1) are barely captured on this 377-gene panel, which is why the automatic annotation missed PT even though PT genes are clearly spatial.

## 9. Single-cell spatial domains

> **Method note — domains at single-cell resolution.** The cell types above say *what* each cell is; a **domain** says *which tissue neighbourhood* it sits in (cortex, medulla, a vascular bundle, an immune aggregate). We use the same BANKSY-style trick as Day 1: build the cell graph, row-normalise it to weights $w_{ij}$, average expression over each cell's neighbours, and stack it onto the cell's own expression,
> $$h_i = \big[\,x_i,\ \lambda\,\bar{x}_{\mathcal N(i)}\,\big],\qquad \bar{x}_{\mathcal N(i)} = \sum_j w_{ij}\,x_j,$$
> then PCA + Leiden. On the 377-gene panel we keep all genes (no HVG step). This is the feature-augmentation route to domains; Day 3 shows the complementary route — clustering each cell's neighbourhood *composition* (BANKSY/CellCharter vs the composition-kmeans niches of Day 3 §4).

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
sq.gr.spatial_neighbors(adata, coord_type='generic', n_neighs=15)
W = normalize(adata.obsp['spatial_connectivities'], norm='l1', axis=1)
X = adata.X
X = np.asarray(X.toarray() if hasattr(X, 'toarray') else X, dtype='float32')
lam = 0.8
H = np.concatenate([X, lam * (W @ X)], axis=1)
Hp = PCA(n_components=30, random_state=0).fit_transform(H)
dom = sc.AnnData(Hp.astype('float32'))
sc.pp.neighbors(dom, n_neighbors=15)
sc.tl.leiden(dom, resolution=0.5, flavor='igraph', n_iterations=2, directed=False, random_state=0)
adata.obs['domain'] = pd.Categorical(dom.obs['leiden'].to_numpy())
print(adata.obs['domain'].nunique(), 'domains')

In [ ]:
sq.pl.spatial_scatter(adata, color='domain', shape=None, size=2)

In [ ]:
pd.crosstab(adata.obs['domain'], adata.obs['celltype'], normalize='index').round(2)

> **Reading the domains.** Each domain is a recurring cell-type *mixture*, read off the crosstab: a row dominated by one type is a homogeneous compartment, a row blending several is an interface or a structured niche (e.g. a glomerulus mixing podocytes, endothelium, and mesangium). Unlike the cell-type map, neighbouring cells of different types can share a domain — that is the point.

> **Exercise (domains).** Re-run the domain step with the neighbour weight $\lambda=0$ (expression only) and compare the map and the domain-by-type crosstab to $\lambda=0.8$. What does adding the neighbourhood term change?

**Solution.**

In [ ]:
H0 = np.concatenate([X, 0.0 * (W @ X)], axis=1)
Hp0 = PCA(n_components=30, random_state=0).fit_transform(H0)
d0 = sc.AnnData(Hp0.astype('float32'))
sc.pp.neighbors(d0, n_neighbors=15)
sc.tl.leiden(d0, resolution=0.5, flavor='igraph', n_iterations=2, directed=False, random_state=0)
adata.obs['domain_l0'] = pd.Categorical(d0.obs['leiden'].to_numpy())
sq.pl.spatial_scatter(adata, color='domain_l0', shape=None, size=2)
print(pd.crosstab(adata.obs['domain_l0'], adata.obs['celltype'], normalize='index').round(2))

**Answer (example).** At $\lambda=0$ the domains track cell type almost one-to-one and look scattered in space; adding the neighbourhood term pulls cells into spatially contiguous domains that mix types by locale, which is what makes them read as tissue regions rather than relabelled cell types.

## 10. Save

In [ ]:
adata.write(RESULTS / 'xenium_kidney_processed.h5ad')

### Recap

You read a large section within budget, applied imaging-specific QC, typed cells on a small panel, and ran single-cell spatial statistics. Day 3 turns these into tissue organisation, communication, and interpretation.